In [0]:
%sql
CREATE OR REPLACE TABLE courier_parcels_delta (
parcel_id INT,
customer_name STRING,
destination_city STRING,
service_type STRING,
weight_kg DOUBLE,
shipping_fee INT,
delivery_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO courier_parcels_delta VALUES
(101, 'Arjun', 'Hyderabad', 'Express', 2.5, 500, 'Delivered'),
(102, 'Meera', 'Mumbai', 'Standard', 4.0, 350, 'In Transit'),
(103, 'Karan', 'Delhi', 'Express', 1.5, 450, 'Pending'),
(104, 'Sneha', 'Pune', 'Standard', 3.0, 300, 'Delivered'),
(105, 'Rohit', 'Chennai', 'Express', 5.0, 800, 'In Transit'),
(106, 'Divya', 'Hyderabad', 'Standard', 2.0, 250, 'Cancelled'),
(107, 'Nikhil', 'Bengaluru', 'Express', 6.0, 950, 'Delivered'),
(108, 'Pooja', 'Mumbai', 'Standard', 1.0, 150, 'Pending'),
(109, 'Vikram', 'Delhi', 'Express', 3.5, 650, 'In Transit'),
(110, 'Anjali', 'Pune', 'Standard', 4.5, 400, 'Cancelled'),
(111, 'Farhan', 'Chennai', 'Express', 2.0, 550, 'Delivered'),
(112, 'Kavya', 'Hyderabad', 'Standard', 5.5, 600, 'Pending'),
(113, 'Manoj', 'Bengaluru', 'Express', 1.5, 400, 'In Transit'),
(114, 'Riya', 'Delhi', 'Standard', 3.0, 300, 'Delivered'),
(115, 'Tarun', 'Mumbai', 'Express', 4.0, 700, 'Pending');


Part A — Creating Delta tables

1. Display all records from courier_parcels_delta .
2. Create another Delta table named courier_parcels_copy using CTAS (CREATE TABLE AS
SELECT).
3. Create a PySpark DataFrame containing three new parcel records and save it as a Delta table
named courier_parcels_pyspark .
4. Verify that all three tables use the Delta format.

In [0]:
%sql
select * from courier_parcels_delta;

In [0]:
%sql

CREATE OR REPLACE TABLE courier_parcels_copy
USING DELTA
AS
SELECT *
FROM courier_parcels_delta;

In [0]:
data = [
    (101, "Arun", "Chennai", "Express", 2.5, 150, "Delivered"),
    (102, "Priya", "Bangalore", "Standard", 5.0, 200, "In Transit"),
    (103, "Karthik", "Mumbai", "Express", 3.2, 250, "Pending")
]

df = spark.createDataFrame(
    data,
    ["parcel_id", "customer_name", "destination_city",
     "service_type", "weight_kg", "shipping_fee", "delivery_status"]
)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("courier_parcels_pyspark")

In [0]:
%sql

DESCRIBE DETAIL courier_parcels_delta;

In [0]:
%sql

DESCRIBE DETAIL courier_parcels_copy;

In [0]:
%sql

DESCRIBE DETAIL courier_parcels_pyspark;

Part B — INSERT, UPDATE and DELETE

5. Insert a new parcel with ID 116, customer Sameer, destination Jaipur, Express service, weight 2.5
kg, shipping fee ₹550, and Pending status.
6. Update the delivery status of parcel 103 to Delivered .
7. Increase the shipping fee of all Express parcels by ₹100.
8. Change the destination city of parcel 108 to Hyderabad.
9. Delete all parcels with Cancelled status.
10. Display the remaining records ordered by shipping fee in descending order.

In [0]:
%sql
insert into courier_parcels_delta values (116, 'Sameer', 'Jaipur', 'Express', 2.5, 550,"Pending");

In [0]:
%sql
update courier_parcels_delta 
set delivery_status="Delivered"
where parcel_id=103;

In [0]:
%sql
update courier_parcels_delta
set shipping_fee=shipping_fee+100 
where service_type="Express";

In [0]:
%sql
update courier_parcels_delta
set destination_city="Hyderabad"
where parcel_id=108;

In [0]:
%sql
delete from courier_parcels_delta
where delivery_status="Cancelled";

In [0]:
%sql
select * from courier_parcels_delta 
order by shipping_fee desc;

Part C — MERGE and UPSERT (SCD Type 1)

11. Display all records from incoming_parcels .
12. Identify which incoming parcel IDs already exist in the target Delta table.
13. Identify which incoming parcel IDs are new.
14. Perform a MERGE operation that updates all columns for existing parcels and inserts new parcels.
15. Verify that parcels 102 and 105 were updated.
16. Verify that parcels 117 and 118 were inserted.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parcels AS
SELECT *
FROM VALUES
(102, 'Meera', 'Mumbai', 'Express', 4.0, 650, 'Delivered'),
(105, 'Rohit', 'Chennai', 'Express', 5.0, 900, 'Delivered'),
(117, 'Ayesha', 'Kolkata', 'Standard', 2.0, 300, 'Pending'),
(118, 'Rahul', 'Jaipur', 'Express', 3.0, 600, 'In Transit')
AS incoming(
parcel_id,
customer_name,
destination_city,
service_type,
weight_kg,
shipping_fee,
delivery_status
);


In [0]:
%sql
select * from incoming_parcels;

In [0]:
%sql

SELECT incoming.parcel_id
FROM incoming_parcels AS incoming
INNER JOIN courier_parcels_delta AS target
ON incoming.parcel_id = target.parcel_id;

In [0]:
%sql
select incoming.parcel_id from incoming_parcels as incoming 
left anti join courier_parcels_delta as target 
on incoming.parcel_id=target.parcel_id;

In [0]:
%sql

MERGE INTO courier_parcels_delta AS target
USING incoming_parcels AS source
ON target.parcel_id = source.parcel_id

WHEN MATCHED THEN
    UPDATE SET
        target.customer_name = source.customer_name,
        target.destination_city = source.destination_city,
        target.service_type = source.service_type,
        target.weight_kg = source.weight_kg,
        target.shipping_fee = source.shipping_fee,
        target.delivery_status = source.delivery_status

WHEN NOT MATCHED THEN
    INSERT (
        parcel_id,
        customer_name,
        destination_city,
        service_type,
        weight_kg,
        shipping_fee,
        delivery_status
    )
    VALUES (
        source.parcel_id,
        source.customer_name,
        source.destination_city,
        source.service_type,
        source.weight_kg,
        source.shipping_fee,
        source.delivery_status
    );

In [0]:
%sql

SELECT *
FROM courier_parcels_delta
WHERE parcel_id IN (102, 105);

In [0]:
%sql

SELECT *
FROM courier_parcels_delta
WHERE parcel_id IN (117, 118);

Part D — Delta table internals and Time Travel

17. Execute DESCRIBE DETAIL and identify the table's format, storage location, number of files,
and size.
18. Execute DESCRIBE HISTORY and identify the operations performed on the table.
19. Find the table version immediately before the MERGE operation.
20. Use VERSION AS OF to retrieve the records before the MERGE.
21. Compare parcel 102 in the previous version with its current record.

In [0]:
%sql
describe detail courier_parcels_delta;

In [0]:
%sql 
describe history courier_parcels_delta;

In [0]:
%sql 
select * from courier_parcels_delta  version as of 8;

In [0]:
%sql

SELECT *
FROM courier_parcels_delta
WHERE parcel_id = 102;

Part E — OPTIMIZE

22. Execute OPTIMIZE on courier_parcels_delta .
23. Run DESCRIBE DETAIL again and compare the number of files before and after optimization.
24. Execute OPTIMIZE courier_parcels_delta ZORDER BY (destination_city) if the
table does not use liquid clustering.
25. Display the transaction history again and identify the optimization operations.

In [0]:
%sql 
optimize courier_parcels_delta;

In [0]:
%sql
describe detail courier_parcels_delta;

In [0]:
%sql

OPTIMIZE courier_parcels_delta
ZORDER BY (destination_city);

In [0]:
%sql

DESCRIBE HISTORY courier_parcels_delta;